In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

In [2]:
import os
import json
import pandas as pd
import random

from datasets import load_dataset

## Load datasets

In [3]:
import requests

def download_file(url, file_path):
    response = requests.get(url)
    response.raise_for_status()

    dir = os.path.dirname(file_path)
    if not os.path.exists(dir):
        os.makedirs(dir)

    with open(file_path, "wb") as file:
        file.write(response.content)

def dump_json(data, file_path):
    dir = os.path.dirname(file_path)
    if not os.path.exists(dir):
        os.makedirs(dir)

    with open(file_path, "w") as file:
        json.dump(data, file, indent=4)

In [4]:
current_dir = os.path.abspath("")
raw_data_dir = os.path.join(current_dir, 'raw')
processed_data_dir = os.path.join(current_dir, 'processed')

# Standard categories from jailbreakbench
STANDARD_CATEGORIES = [
    "Harassment/Discrimination",
    "Malware/Hacking",
    "Physical harm",
    "Economic harm",
    "Fraud/Deception",
    "Disinformation",
    "Sexual/Adult content",
    "Privacy",
    "Expert advice",
    "Government decision-making"
]

def process_advbench():
    """Process advbench.csv - already categorized"""
    raw_file_path = os.path.join(raw_data_dir, 'advbench.csv')
    processed_file_path = os.path.join(processed_data_dir, 'advbench.json')

    if not os.path.exists(raw_file_path):
        raise FileNotFoundError(f"Raw file not found: {raw_file_path}. Please run categorize_datasets.py first.")
    
    dataset = pd.read_csv(raw_file_path)
    instructions = dataset['goal'].to_list()
    categories = dataset['Category'].to_list()

    dataset_json = [{'instruction': instruction.strip(), 'category': category} for instruction, category in zip(instructions, categories)]
    dump_json(dataset_json, processed_file_path)

def process_malicious_instruct():
    """Process malicious_instruct.csv - already categorized"""
    raw_file_path = os.path.join(raw_data_dir, 'malicious_instruct.csv')
    processed_file_path = os.path.join(processed_data_dir, 'malicious_instruct.json')

    if not os.path.exists(raw_file_path):
        raise FileNotFoundError(f"Raw file not found: {raw_file_path}. Please run categorize_datasets.py first.")
    
    dataset = pd.read_csv(raw_file_path)
    instructions = dataset['text'].to_list()
    categories = dataset['Category'].to_list()

    dataset_json = [{'instruction': instruction.strip(), 'category': category} for instruction, category in zip(instructions, categories)]
    dump_json(dataset_json, processed_file_path)

def process_tdc2023():
    """Process tdc2023 JSON files - already categorized"""
    raw_file_paths = [
        os.path.join(raw_data_dir, 'tdc2023_dev_behaviors.json'),
        os.path.join(raw_data_dir, 'tdc2023_test_behaviors.json')
    ]
    processed_file_path = os.path.join(processed_data_dir, 'tdc2023.json')

    for raw_file_path in raw_file_paths:
        if not os.path.exists(raw_file_path):
            raise FileNotFoundError(f"Raw file not found: {raw_file_path}. Please run categorize_datasets.py first.")

    dataset_json = []
    for raw_file_path in raw_file_paths:
        with open(raw_file_path, 'r') as f:
            data = json.load(f)
        for item in data:
            dataset_json.append({
                'instruction': item['text'].strip(),
                'category': item['category']
            })

    dump_json(dataset_json, processed_file_path)

def process_jailbreakbench():
    """Process jailbreakbench.csv - already has Category column"""
    raw_file_path = os.path.join(raw_data_dir, 'jailbreakbench.csv')
    processed_file_path = os.path.join(processed_data_dir, 'jailbreakbench.json')

    if not os.path.exists(raw_file_path):
        raise FileNotFoundError(f"Raw file not found: {raw_file_path}")
    
    dataset = pd.read_csv(raw_file_path)
    instructions = dataset['Goal'].to_list()
    categories = dataset['Category'].to_list()

    dataset_json = [{'instruction': instruction.strip(), 'category': category} for instruction, category in zip(instructions, categories)]
    dump_json(dataset_json, processed_file_path)

def process_harmbench(split):
    """Process harmbench CSV files - already categorized"""
    assert split in ['val', 'test']
    
    raw_file_path = os.path.join(raw_data_dir, f'harmbench_{split}.csv')
    processed_file_path = os.path.join(processed_data_dir, f'harmbench_{split}.json')

    if not os.path.exists(raw_file_path):
        raise FileNotFoundError(f"Raw file not found: {raw_file_path}. Please run categorize_datasets.py first.")
    
    dataset = pd.read_csv(raw_file_path)

    # Filter out instructions with FunctionalCategory=copyright or Tags=context
    filtered_dataset = dataset[
        ~dataset['FunctionalCategory'].str.contains('copyright', case=False, na=False) &
        ~dataset['Tags'].str.contains('context', case=False, na=False)
    ]

    instructions = filtered_dataset['Behavior'].to_list()
    categories = filtered_dataset['Category'].to_list()  # Use the new Category column

    dataset_json = [{'instruction': instruction.strip(), 'category': category} for instruction, category in zip(instructions, categories)]
    dump_json(dataset_json, processed_file_path)

def process_strongreject():
    """Process strongreject.csv - already categorized"""
    raw_file_path = os.path.join(raw_data_dir, 'strongreject.csv')
    processed_file_path = os.path.join(processed_data_dir, 'strongreject.json')

    if not os.path.exists(raw_file_path):
        raise FileNotFoundError(f"Raw file not found: {raw_file_path}. Please run categorize_datasets.py first.")
    
    dataset = pd.read_csv(raw_file_path)
    instructions = dataset['forbidden_prompt'].to_list()
    categories = dataset['Category'].to_list()  # Use the new Category column

    dataset_json = [{'instruction': instruction.strip(), 'category': category} for instruction, category in zip(instructions, categories)]
    dump_json(dataset_json, processed_file_path)

def process_alpaca():
    """Process alpaca dataset - harmless dataset"""
    hf_path = 'tatsu-lab/alpaca'
    processed_file_path = os.path.join(processed_data_dir, 'alpaca.json')

    dataset = load_dataset(hf_path)

    # filter for instructions that do not have inputs
    instructions = []
    for i in range(len(dataset['train'])):
        if dataset['train'][i]['input'].strip() == '':
            instructions.append(dataset['train'][i]['instruction'])

    dataset_json = [{'instruction': instruction.strip(), 'category': None} for instruction in instructions]
    dump_json(dataset_json, processed_file_path)

In [5]:
# Process all datasets from categorized raw files
process_advbench()
process_malicious_instruct()
process_tdc2023()

process_jailbreakbench()
process_harmbench(split='val')
process_harmbench(split='test')
process_strongreject()

process_alpaca()

print("All datasets processed successfully!")

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

All datasets processed successfully!


## Construct train, val, test datasets

In [17]:
current_dir = os.path.abspath("")
splits_data_dir = os.path.join(current_dir, 'splits')

max_train_subset_size = 128 # limits the number of examples from a single dataset

def filter_by_categories(data, allowed_categories=None):
    """
    Filter dataset by allowed categories.
    
    Args:
        data: List of dicts with 'category' field
        allowed_categories: List of category names to include, or None to include all
    
    Returns:
        Filtered dataset
    """
    if allowed_categories is None:
        return data
    
    # Normalize category names (case-insensitive matching)
    allowed_categories_lower = [cat.lower() for cat in allowed_categories]
    
    filtered_data = []
    for item in data:
        category = item.get('category')
        if category is None:
            # Include items with no category if explicitly allowed
            if None in allowed_categories or 'None' in allowed_categories:
                filtered_data.append(item)
        elif category.lower() in allowed_categories_lower:
            filtered_data.append(item)
    
    return filtered_data

def get_category_distribution(data):
    """Get distribution of categories in the dataset."""
    from collections import Counter
    categories = [item.get('category', 'None') for item in data]
    return Counter(categories)

def construct_harmful_dataset_splits(allowed_categories=None, train_categories=None, val_categories=None, test_categories=None):
    """
    Construct harmful dataset splits by combining all datasets, shuffling, and splitting.
    
    Args:
        allowed_categories: List of category names to include in all splits, or None to include all categories.
                           If provided, train_categories/val_categories/test_categories will be intersected with this.
        train_categories: List of category names to include in train split, or None to include all categories
        val_categories: List of category names to include in val split, or None to include all categories
        test_categories: List of category names to include in test split, or None to include all categories
    
    Note: If allowed_categories is provided, it acts as a global filter that applies to all splits.
          train_categories/val_categories/test_categories will be intersected with allowed_categories.
    """
    harmful_train_path = os.path.join(splits_data_dir, 'harmful_train.json')
    harmful_val_path = os.path.join(splits_data_dir, 'harmful_val.json')
    harmful_test_path = os.path.join(splits_data_dir, 'harmful_test.json')

    # Combine all harmful datasets
    all_harmful_instructions = []
    dataset_files = [
        'advbench.json',
        'malicious_instruct.json',
        'tdc2023.json',
        'harmbench_val.json',
        'harmbench_test.json',
        'jailbreakbench.json',
        'strongreject.json'
    ]
    
    print("Loading all harmful datasets...")
    for file in dataset_files:
        file_path = os.path.join(processed_data_dir, file)
        if os.path.exists(file_path):
            with open(file_path, 'r') as f:
                data = json.load(f)
                all_harmful_instructions.extend(data)
                print(f"  Loaded {len(data)} examples from {file}")
        else:
            print(f"  Warning: {file} not found, skipping")
    
    # Remove duplicates based on instruction text
    seen_instructions = set()
    unique_instructions = []
    duplicates_removed = 0
    
    for instruction in all_harmful_instructions:
        inst_text = instruction['instruction']
        if inst_text not in seen_instructions:
            seen_instructions.add(inst_text)
            unique_instructions.append(instruction)
        else:
            duplicates_removed += 1
    
    print(f"\nRemoved {duplicates_removed} duplicate instructions")
    print(f"Total unique instructions: {len(unique_instructions)}")
    
    # Apply global filter if specified
    if allowed_categories is not None:
        unique_instructions = filter_by_categories(unique_instructions, allowed_categories)
        print(f"After global category filter: {len(unique_instructions)} instructions")
    
    # Randomly shuffle the combined dataset
    random.seed(42)
    random.shuffle(unique_instructions)
    
    # Determine category filters for each split
    # If global filter exists, intersect with split-specific filters
    if allowed_categories is not None:
        if train_categories is not None:
            train_categories = [cat for cat in train_categories if cat in allowed_categories]
        else:
            train_categories = allowed_categories
            
        if val_categories is not None:
            val_categories = [cat for cat in val_categories if cat in allowed_categories]
        else:
            val_categories = allowed_categories
            
        if test_categories is not None:
            test_categories = [cat for cat in test_categories if cat in allowed_categories]
        else:
            test_categories = allowed_categories
    
    # Filter each split separately
    train_candidates = filter_by_categories(unique_instructions, train_categories)
    val_candidates = filter_by_categories(unique_instructions, val_categories)
    test_candidates = filter_by_categories(unique_instructions, test_categories)
    
    print(f"\nCategory filtering:")
    if train_categories:
        print(f"  Train categories: {train_categories} -> {len(train_candidates)} candidates")
    else:
        print(f"  Train categories: All -> {len(train_candidates)} candidates")
    if val_categories:
        print(f"  Val categories: {val_categories} -> {len(val_candidates)} candidates")
    else:
        print(f"  Val categories: All -> {len(val_candidates)} candidates")
    if test_categories:
        print(f"  Test categories: {test_categories} -> {len(test_candidates)} candidates")
    else:
        print(f"  Test categories: All -> {len(test_candidates)} candidates")
    
    # Split in the same proportion as harmless dataset (60/20/20)
    train_p, val_p, test_p = 0.6, 0.20, 0.20
    
    # Sample from candidates for each split
    random.seed(42)  # Reset seed for reproducible sampling
    
    train_size = int(train_p * len(train_candidates)) if len(train_candidates) > 0 else 0
    val_size = int(val_p * len(val_candidates)) if len(val_candidates) > 0 else 0
    test_size = int(test_p * len(test_candidates)) if len(test_candidates) > 0 else 0
    
    # Shuffle candidates before sampling
    random.shuffle(train_candidates)
    random.shuffle(val_candidates)
    random.shuffle(test_candidates)
    
    harmful_train_instructions = train_candidates[:train_size] if train_size > 0 else []
    harmful_val_instructions = val_candidates[:val_size] if val_size > 0 else []
    harmful_test_instructions = test_candidates[:test_size] if test_size > 0 else []
    
    # Print category distribution
    print("\nCategory distribution:")
    print("Train:", get_category_distribution(harmful_train_instructions))
    print("Val:", get_category_distribution(harmful_val_instructions))
    print("Test:", get_category_distribution(harmful_test_instructions))

    dump_json(harmful_train_instructions, harmful_train_path)
    dump_json(harmful_val_instructions, harmful_val_path)
    dump_json(harmful_test_instructions, harmful_test_path)
    
    print(f"\nSaved datasets:")
    if len(train_candidates) > 0:
        print(f"  Train: {len(harmful_train_instructions)} examples ({len(harmful_train_instructions)/len(train_candidates)*100:.1f}% of train candidates)")
    else:
        print(f"  Train: {len(harmful_train_instructions)} examples")
    if len(val_candidates) > 0:
        print(f"  Val: {len(harmful_val_instructions)} examples ({len(harmful_val_instructions)/len(val_candidates)*100:.1f}% of val candidates)")
    else:
        print(f"  Val: {len(harmful_val_instructions)} examples")
    if len(test_candidates) > 0:
        print(f"  Test: {len(harmful_test_instructions)} examples ({len(harmful_test_instructions)/len(test_candidates)*100:.1f}% of test candidates)")
    else:
        print(f"  Test: {len(harmful_test_instructions)} examples")

def construct_harmless_dataset_splits():
    harmless_train_path = os.path.join(splits_data_dir, 'harmless_train.json')
    harmless_val_path = os.path.join(splits_data_dir, 'harmless_val.json')
    harmless_test_path = os.path.join(splits_data_dir, 'harmless_test.json')

    train_p, val_p, test_p = 0.6, 0.20, 0.20

    harmless_instructions = []
    for file in ['alpaca.json']:
        with open(os.path.join(processed_data_dir, file), 'r') as f:
            harmless_instructions.extend(json.load(f))

    random.seed(42)
    random.shuffle(harmless_instructions)

    total_size = len(harmless_instructions)
    train_size = int(train_p * total_size)
    val_size = int(val_p * total_size)

    harmless_train_instructions = harmless_instructions[:train_size]
    harmless_val_instructions = harmless_instructions[train_size:train_size+val_size]
    harmless_test_instructions = harmless_instructions[train_size+val_size:]

    dump_json(harmless_train_instructions, harmless_train_path)
    dump_json(harmless_val_instructions, harmless_val_path)
    dump_json(harmless_test_instructions, harmless_test_path)
    
    print(f"\nSaved harmless datasets:")
    print(f"  Train: {len(harmless_train_instructions)} examples")
    print(f"  Val: {len(harmless_val_instructions)} examples")
    print(f"  Test: {len(harmless_test_instructions)} examples")

In [18]:
ALLOWED_CATEGORIES = [
    "Harassment/Discrimination",
    "Malware/Hacking",
    "Physical harm",
    "Economic harm",
    "Fraud/Deception",
    "Disinformation",
    "Sexual/Adult content",
    "Privacy",
    "Expert advice",
    "Government decision-making"
]
construct_harmful_dataset_splits(ALLOWED_CATEGORIES)
construct_harmless_dataset_splits()

Loading all harmful datasets...
  Loaded 520 examples from advbench.json
  Loaded 100 examples from malicious_instruct.json
  Loaded 100 examples from tdc2023.json
  Loaded 41 examples from harmbench_val.json
  Loaded 159 examples from harmbench_test.json
  Loaded 100 examples from jailbreakbench.json
  Loaded 313 examples from strongreject.json

Removed 102 duplicate instructions
Total unique instructions: 1231
After global category filter: 1231 instructions

Category filtering:
  Train categories: ['Harassment/Discrimination', 'Malware/Hacking', 'Physical harm', 'Economic harm', 'Fraud/Deception', 'Disinformation', 'Sexual/Adult content', 'Privacy', 'Expert advice', 'Government decision-making'] -> 1231 candidates
  Val categories: ['Harassment/Discrimination', 'Malware/Hacking', 'Physical harm', 'Economic harm', 'Fraud/Deception', 'Disinformation', 'Sexual/Adult content', 'Privacy', 'Expert advice', 'Government decision-making'] -> 1231 candidates
  Test categories: ['Harassment/Dis

## Generate datasets with specific categories

You can generate datasets filtered by specific categories. The function supports:
- **Global filtering**: Use `allowed_categories` to filter all splits
- **Split-specific filtering**: Use `train_categories`, `val_categories`, `test_categories` to filter each split independently
- **Combined**: Use `allowed_categories` as a global filter and `train_categories`/`val_categories`/`test_categories` as additional filters that will be intersected

### Available categories:
- Harassment/Discrimination
- Malware/Hacking
- Physical harm
- Economic harm
- Fraud/Deception
- Disinformation
- Sexual/Adult content
- Privacy
- Expert advice
- Government decision-making


# Example: Generate datasets with different categories for each split
# Uncomment one of the examples below:

# Example 1: Same categories for all splits (backward compatible)
# construct_harmful_dataset_splits(allowed_categories=[
#     "Physical harm",
#     "Malware/Hacking",
#     "Sexual/Adult content"
# ])

# Example 2: Different categories for each split
# construct_harmful_dataset_splits(
#     train_categories=["Harassment/Discrimination", "Disinformation"],
#     val_categories=["Malware/Hacking", "Physical harm"],
#     test_categories=["Sexual/Adult content", "Privacy", "Economic harm"]
# )

# Example 3: Global filter + split-specific filters
# Train only on high-risk categories, validate on info-related, test on all globally allowed
# construct_harmful_dataset_splits(
#     allowed_categories=["Physical harm", "Malware/Hacking", "Disinformation", "Fraud/Deception"],
#     train_categories=["Physical harm", "Malware/Hacking"],  # Only train on high-risk
#     val_categories=["Disinformation"],  # Validate on info-related
#     test_categories=None  # Test on all globally allowed categories
# )

# Example 4: Train on single category, test on all categories
# construct_harmful_dataset_splits(
#     train_categories=["Disinformation"],
#     val_categories=["Disinformation"],
#     test_categories=None  # All categories
# )

print("Available categories:")
for cat in STANDARD_CATEGORIES:
    print(f"  - {cat}")
